# 01 — Coleta

Baixa as políticas de privacidade dos fabricantes do inventário e levanta os projetos de lei sobre dados neurais na Câmara e no Senado.

Tudo vai para `data/raw/` exatamente como veio do servidor. As funções estão em `src/coleta.py` e `src/projetos_lei.py`; este notebook só as executa e mostra o resultado.

In [1]:
from pathlib import Path
import os
import sys

# Os caminhos do projeto são relativos à raiz do repositório.
RAIZ = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))

import pandas as pd

pd.set_option("display.max_colwidth", 90)

## Políticas de privacidade

A lista de documentos a coletar fica em `data/raw/fontes_politicas.csv`. Cada fabricante tem um documento principal e, em alguns casos, documentos complementares (por exemplo, uma política separada para dados de saúde).

In [2]:
fontes = pd.read_csv("data/raw/fontes_politicas.csv")
fontes.groupby("tipo").size().rename("documentos").to_frame()

,documentos
tipo,
complementar,6
principal,15


A coleta salva cada página com a data no nome do arquivo e registra status, tamanho e hash em `data/raw/politicas/coletas.csv`. Documentos já baixados na mesma data são pulados, então a célula pode ser executada de novo sem repetir os downloads.

In [3]:
from src.coleta import coletar_politicas, ler_registro

novos = coletar_politicas()
print(f"{len(novos)} documentos baixados nesta execução")

registro = pd.DataFrame(ler_registro())
registro[["id_documento", "status_http", "data_hora_utc", "bytes", "url_final"]]

0 documentos baixados nesta execução


,id_documento,status_http,data_hora_utc,bytes,url_final
0,P01-1,200,2026-11-15T04:42:40+00:00,619861,https://choosemuse.com/pages/legal
1,P01-2,200,2026-11-15T04:42:46+00:00,279934,https://choosemuse.com/policies/privacy-policy
2,P02-1,200,2026-11-15T04:42:53+00:00,126571,https://neurosky.com/privacy-policy/
3,P03-1,200,2026-11-15T04:43:00+00:00,55725,https://id.emotivcloud.com/eoidc/privacy/privacy_policy/
4,P03-2,200,2026-11-15T04:43:06+00:00,777031,https://www.emotiv.com/consumer-health-data-privacy-policy
5,P04-1,200,2026-11-15T04:43:13+00:00,256816,https://www.meetflowtime.com/policies/privacy-policy
6,P05-1,200,2026-11-15T04:43:20+00:00,76950,https://brainbit.com/privacy_policy/
7,P06-1,200,2026-11-15T04:43:27+00:00,18867,https://api.macrotellect.com/privacy/index
8,P07-1,200,2026-11-15T04:43:34+00:00,314648,https://neurosity.co/privacy-policy
9,P07-2,200,2026-11-15T04:43:40+00:00,281291,https://neurosity.co/privacy


In [4]:
falhas = registro[registro["status_http"] != "200"]
print(f"{len(falhas)} coletas com falha")
falhas

0 coletas com falha


,id_documento,url,url_final,status_http,data_hora_utc,arquivo,bytes,sha256


## Projetos de lei

A busca usa as APIs de Dados Abertos da Câmara e do Senado, com os termos definidos em `src/projetos_lei.py`. A busca por palavra-chave também traz proposições que só usam o termo de passagem; a coluna `trata_do_tema` separa as que falam de fato de dados ou direitos ligados ao cérebro.

As respostas das APIs ficam em `data/raw/projetos_lei/` e a tabela consolidada em `data/interim/projetos_lei.csv`.

In [5]:
from src.projetos_lei import TERMOS, montar_tabela

print("Termos buscados:", ", ".join(TERMOS))
projetos = pd.DataFrame(montar_tabela())
projetos.groupby(["casa", "trata_do_tema"]).size().rename("proposições").to_frame()

Termos buscados: dados neurais, neural, neurodireitos, neurotecnologia, integridade mental, atividade cerebral


proposições
casa   trata_do_tema             
Câmara nao                      2
       sim                      5
Senado nao                      2
       sim                      1

In [6]:
projetos.loc[projetos["trata_do_tema"] == "sim", ["casa", "sigla", "numero", "ano", "situacao", "autoria", "ementa"]]

,casa,sigla,numero,ano,situacao,autoria,ementa
0,Câmara,PL,1221,2021,Retirado pelo(a) Autor(a),Carlos Henrique Gaguim,"Modifica a Lei n° 13.709, de 14 de agosto de 2018 (Lei Geral de Proteção de Dados Pess..."
1,Câmara,PL,1229,2021,Aguardando Encaminhamento,Carlos Henrique Gaguim,"Modifica a Lei n° 13.709, de 14 de agosto de 2018 (Lei Geral de Proteção de Dados Pess..."
3,Câmara,PL,522,2022,Aguardando Parecer,Carlos Henrique Gaguim,"Modifica a Lei n° 13.709, de 14 de agosto de 2018 (Lei Geral de Proteção de Dados Pess..."
4,Câmara,PL,2174,2023,Aguardando Designação de Relator(a),Rubens Pereira Júnior,Estabelece as normas e princípios para proteção dos direitos fundamentais relacionados...
6,Câmara,PL,6687,2025,Tramitando em Conjunto,Amom Mandel,Altera a Lei nº 13.709/2018 para incluir os dados neurais no rol de dados pessoais sen...
7,Senado,PEC,29,2023,,"Senador Randolfe Rodrigues (REDE/AP), Senador Weverton (PDT/MA), Senador Styvenson Val...","Altera a Constituição Federal para incluir, entre os direitos e garantias fundamentais..."


A PEC 29/2023 do Senado entra por inclusão manual: a busca por palavra-chave daquela API não a devolve, embora a ementa trate de integridade mental. As matérias incluídas dessa forma estão listadas em `MATERIAS_SENADO`.